# Gemini Trojan Detection — Full Pipeline
Runs on Kaggle T4 GPU. Covers: train clean + poisoned models → generate 10-signal features → train meta-classifier → demo audit.

In [ ]:
# ── 1. Install deps (skip torch/torchvision — Kaggle pre-installs them) ──
import subprocess, sys

DEPS = [
    'fastapi==0.134.0',
    'scikit-learn==1.8.0',
    'captum==0.7.0',
    'onnx==1.20.1',
    'onnx2torch==1.5.15',
    'onnxruntime==1.24.2',
    'opencv-python-headless==4.10.0.84',
    'tabulate==0.9.0',
    'tqdm==4.67.3',
]

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + DEPS)
print('Deps installed.')

In [ ]:
# ── 2. Mount project code ──
# Option A: repo is a Kaggle dataset → add it via "Add Data" and set PROJECT_ROOT below.
# Option B: public GitHub repo → uncomment the clone line.

import os, sys

PROJECT_ROOT = '/kaggle/input/gemini-trojan-detection'  # adjust if dataset name differs
# subprocess.check_call(['git', 'clone', 'https://github.com/YOUR/REPO.git', '/kaggle/working/project'])
# PROJECT_ROOT = '/kaggle/working/project'

if not os.path.exists(PROJECT_ROOT):
    # Fallback: copy from working dir if user uploaded manually
    PROJECT_ROOT = '/kaggle/working'

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print(f'Working from: {os.getcwd()}')
print('Files:', [f for f in os.listdir('.') if f.endswith('.py')])

In [ ]:
# ── 3. Verify GPU ──
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── 4. Train clean + poisoned models ──
import torch.nn as nn
import torch.optim as optim

from dataset import get_cifar10_dataloaders
from models import get_resnet18

EPOCHS = 10          # increase for better accuracy; 10 is good balance on T4
BATCH_SIZE = 256     # T4 handles 256 comfortably
POISON_RATIO = 0.1
TARGET_CLASS = 0
TRIGGER_TYPE = 'checkerboard'

os.makedirs('models', exist_ok=True)
criterion = nn.CrossEntropyLoss()

def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for inputs, labels, *_ in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(inputs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += out.argmax(1).eq(labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), 100. * correct / total

@torch.no_grad()
def evaluate(model, loader, label='Test'):
    model.eval()
    correct, total = 0, 0
    for inputs, labels, *_ in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        correct += model(inputs).argmax(1).eq(labels).sum().item()
        total += labels.size(0)
    acc = 100. * correct / total
    print(f'  {label}: {acc:.2f}%')
    return acc

def run_training(name, train_loader, test_clean, test_poisoned=None, save_path=None):
    print(f'\n=== Training {name} ===')
    model = get_resnet18(num_classes=10).to(device)
    opt = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    for epoch in range(1, EPOCHS + 1):
        loss, acc = train_one_epoch(model, train_loader, opt)
        sched.step()
        print(f'  Epoch {epoch}/{EPOCHS} — loss: {loss:.4f}, train acc: {acc:.1f}%')
    evaluate(model, test_clean, 'Clean Acc')
    if test_poisoned:
        evaluate(model, test_poisoned, 'ASR (Attack Success Rate)')
    if save_path:
        torch.save(model.state_dict(), save_path)
        print(f'  Saved → {save_path}')
    return model

# Clean model
train_clean, test_clean, _ = get_cifar10_dataloaders(BATCH_SIZE, poison_ratio=0.0)
clean_model = run_training('CLEAN', train_clean, test_clean, save_path='models/clean_model.pth')

# Poisoned model
train_poison, test_clean_p, test_poisoned_p = get_cifar10_dataloaders(
    BATCH_SIZE, poison_ratio=POISON_RATIO, target_class=TARGET_CLASS, trigger_type=TRIGGER_TYPE
)
poisoned_model = run_training(
    f'POISONED (ratio={POISON_RATIO}, trigger={TRIGGER_TYPE})',
    train_poison, test_clean_p, test_poisoned_p,
    save_path='models/poisoned_model.pth'
)

In [ ]:
# ── 5. (Optional) Train additional poisoned variants for richer meta-classifier dataset ──
VARIANTS = [
    ('square',      0.05, 1),
    ('blending',    0.15, 2),
    ('clean_label', 0.10, 3),
]

for trigger, ratio, target in VARIANTS:
    tl, tc, tp = get_cifar10_dataloaders(BATCH_SIZE, poison_ratio=ratio, target_class=target, trigger_type=trigger)
    run_training(
        f'POISONED-{trigger}',
        tl, tc, tp,
        save_path=f'models/poisoned_{trigger}_r{int(ratio*100)}_t{target}.pth'
    )

print('\nAll models trained.')
print('models/', os.listdir('models'))

In [ ]:
# ── 6. Generate 10-signal feature vectors (meta-classifier training data) ──
# Runs defenses directly — no Celery/Redis needed on Kaggle.
from train_meta_classifier import generate_training_data

X, y = generate_training_data(model_dir='models', output_file='meta_training_data.npz')
print(f'\nFeature matrix: {X.shape}, Labels: {y.shape}')
print(f'Poisoned: {y.sum()}, Clean: {(y == 0).sum()}')

In [ ]:
# ── 7. Train meta-classifier ──
from train_meta_classifier import train_meta_classifier

train_meta_classifier(data_file='meta_training_data.npz')
print('meta_classifier.pkl saved.')

In [ ]:
# ── 8. Demo audit — run all 10 signals on both models ──
import numpy as np
from defenses import (
    NeuralCleanse, STRIP, ActivationClustering, WeightAnalysis,
    NaturalTrojanProfiler, GradientSimilarity, SpectralSignatures,
    ConfidenceDistributionAnalysis, RiskFusionEngine,
)
from trojai_model_wrapper import TrojAI_ModelWrapper

engine = RiskFusionEngine(use_meta_classifier=True)

def quick_audit(model_path, label_str):
    print(f'\n=== Auditing {label_str} ({os.path.basename(model_path)}) ===')
    state_dict = torch.load(model_path, map_location=device, weights_only=False)
    raw = get_resnet18(num_classes=10)
    raw.load_state_dict(state_dict)
    model = TrojAI_ModelWrapper(raw, device)
    model.eval()

    train_d, test_c, test_p = get_cifar10_dataloaders(64, poison_ratio=0.1, target_class=0, trigger_type='checkerboard')
    signals = {}

    try:
        nc = NeuralCleanse(model, device, num_classes=10)
        flagged, sizes, masks, patterns, anomaly_idx = nc.detect(test_c, epochs=3, target_class=0)
        signals['neural_cleanse'] = engine.normalize_neural_cleanse(anomaly_idx.tolist() if len(anomaly_idx) else [])
    except Exception as e:
        signals['neural_cleanse'] = 0.0; print(f'  NC error: {e}')

    try:
        strip = STRIP(model, device, test_c.dataset)
        n = min(20, len(test_c.dataset))
        ce = [strip.calculate_entropy(test_c.dataset[i][0].to(device), num_samples=32) for i in range(n)]
        pe = [strip.calculate_entropy(test_p.dataset[i][0].to(device), num_samples=32) for i in range(n)]
        thr = np.percentile(ce, 5)
        signals['strip'] = engine.normalize_strip(
            sum(1 for e in pe if e >= thr) / len(pe),
            sum(1 for e in ce if e < thr) / len(ce)
        )
    except Exception as e:
        signals['strip'] = 0.0; print(f'  STRIP error: {e}')

    try:
        ac = ActivationClustering(model, device, feature_layer_name=model.feature_layer_name)
        score, *_ = ac.detect(train_d, target_class=0, include_tsne=False, include_secondary_layer=False)
        signals['activation_clustering'] = engine.normalize_clustering(score)
        ac.remove_hook()
    except Exception as e:
        signals['activation_clustering'] = 0.0; print(f'  AC error: {e}')

    try:
        wa = WeightAnalysis(model, device)
        signals['weight_analysis'] = engine.normalize_weight_analysis(wa.detect())
    except Exception as e:
        signals['weight_analysis'] = 0.0; print(f'  WA error: {e}')

    try:
        ntp = NaturalTrojanProfiler(model, device)
        signals['ntp'] = min(ntp.profile_shortcuts(test_c, num_batches=6) * 1.5, 1.0)
    except Exception as e:
        signals['ntp'] = 0.0; print(f'  NTP error: {e}')

    try:
        gs = GradientSimilarity(model, device)
        signals['gradient'] = engine.normalize_gradient_similarity(gs.detect(test_c, target_class=0, num_samples=12))
    except Exception as e:
        signals['gradient'] = 0.0; print(f'  Grad error: {e}')

    try:
        sp = SpectralSignatures(model, device)
        res = sp.detect(train_d, target_class=0)
        signals['spectral'] = engine.normalize_spectral_signatures(float(res[3])) if isinstance(res, tuple) and len(res) >= 4 else 0.0
        sp.remove_hook()
    except Exception as e:
        signals['spectral'] = 0.0; print(f'  Spectral error: {e}')

    try:
        cda = ConfidenceDistributionAnalysis(model, device)
        raw_risk, _ = cda.detect(test_c, target_class=0, num_batches=5)
        signals['cda'] = min(max(float(raw_risk), 0.0), 1.0)
    except Exception as e:
        signals['cda'] = 0.0; print(f'  CDA error: {e}')

    feature_vec = [0.0, 0.0,  # blackbox + behavioral (need loaders to compute; skip in demo)
                   signals.get('neural_cleanse', 0.0),
                   signals.get('strip', 0.0),
                   signals.get('activation_clustering', 0.0),
                   signals.get('weight_analysis', 0.0),
                   signals.get('ntp', 0.0),
                   signals.get('gradient', 0.0),
                   signals.get('spectral', 0.0),
                   signals.get('cda', 0.0)]

    fused = engine.fuse(feature_vec)
    print(f'  Signals: {{\'nc\': {signals["neural_cleanse"]:.3f}, \'strip\': {signals["strip"]:.3f}, \'ac\': {signals["activation_clustering"]:.3f}}}')
    print(f'  Fused risk score: {fused:.3f}')
    return fused

clean_score   = quick_audit('models/clean_model.pth',   'CLEAN MODEL')
poisoned_score = quick_audit('models/poisoned_model.pth', 'POISONED MODEL')

print(f'\nSummary:')
print(f'  Clean model risk:    {clean_score:.3f}')
print(f'  Poisoned model risk: {poisoned_score:.3f}')
print(f'  Delta: {poisoned_score - clean_score:.3f} (higher = better separation)')

In [ ]:
# ── 9. Save outputs to /kaggle/working for download ──
import shutil

OUT = '/kaggle/working/outputs'
os.makedirs(OUT, exist_ok=True)

for f in ['meta_classifier.pkl', 'meta_training_data.npz']:
    if os.path.exists(f):
        shutil.copy(f, OUT)

shutil.copytree('models', f'{OUT}/models', dirs_exist_ok=True)

print('Outputs saved to /kaggle/working/outputs:')
for root, _, files in os.walk(OUT):
    for f in files:
        path = os.path.join(root, f)
        print(f'  {path} ({os.path.getsize(path)/1e6:.1f} MB)')